# Daily mean positions from regular GNSS

This notebook reads processed GNSS `.pos` files and calculates daily mean latitude, longitude, and height for the 29 paper stations over 2011–2026. It writes one daily-position CSV per station and plots station tracks and horizontal displacement.

Input files are expected under `GNSS_data/<year>/<station>/*.pos`. This notebook does not download or process RINEX files.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# ──────────────────────────────────────────────────────────────────────────────
# CONFIG
# ──────────────────────────────────────────────────────────────────────────────
# Point this to the folder containing year/station .pos files from GNSS processing.
base_dir = Path('/Volumes/icefall/tinydrive/GNSS_data')   # layout: /YYYY/<station>/*.pos
STATIONS = ['gz01', 'gz02', 'gz06', 'gz09', 'gz13', 'gz17', 'gz18', 'la01', 'la02', 'la03', 'la04', 'la05', 'la06', 'la07', 'la08', 'la09', 'la10', 'la11', 'la12', 'la13', 'la14', 'la15', 'la16', 'la17', 'la18', 'mg04', 'mg05', 'mg06', 'mg07']
YEAR_START, YEAR_END = 2011, 2026

OUT_CSV_DIR = base_dir / 'daily_positions'
OUT_FIG_DIR = base_dir / 'figs'
OUT_CSV_DIR.mkdir(exist_ok=True, parents=True)
OUT_FIG_DIR.mkdir(exist_ok=True, parents=True)

pd.options.mode.copy_on_write = True

# ──────────────────────────────────────────────────────────────────────────────
# Helpers
# ──────────────────────────────────────────────────────────────────────────────
def dms_to_decimal(dd, mn, ss):
    """Convert DMS to signed decimal degrees using sign of degrees."""
    dd = pd.to_numeric(dd, errors='coerce')
    mn = pd.to_numeric(mn, errors='coerce')
    ss = pd.to_numeric(ss, errors='coerce')
    sign = np.sign(dd)
    sign = np.where(sign == 0, 1.0, sign)
    return sign * (np.abs(dd) + mn/60.0 + ss/3600.0)

def read_pos_file(pos_path, station_hint=None):
    """
    Parse one .pos using your schema. Returns columns:
    ['STN','time','lat_deg','lon_deg','HGT(m)']
    """
    try:
        lines = pos_path.read_text(encoding='latin-1', errors='ignore').splitlines()
        if len(lines) < 4:
            return pd.DataFrame()

        cols = lines[3].strip().split()
        # Require expected DMS fields & time fields
        required = {'YEAR-MM-DD','HR:MN:SS.SS','LATDD','LATMN','LATSS','LONDD','LONMN','LONSS','HGT(m)'}
        if not required.issubset(set(cols)):
            # Load everything anyway just so we can inspect if needed
            df = pd.read_csv(pos_path, sep=r'\s+', skiprows=4, names=cols, engine='python')
            print(f"⚠️  {pos_path.name}: missing expected columns, got {cols}")
            return pd.DataFrame()

        df = pd.read_csv(
            pos_path, sep=r'\s+', skiprows=4, names=cols, engine='python',
            usecols=list(required | {'STN'})
        )

        # Decimal degrees
        df['lat_deg'] = dms_to_decimal(df['LATDD'], df['LATMN'], df['LATSS'])
        df['lon_deg'] = dms_to_decimal(df['LONDD'], df['LONMN'], df['LONSS'])

        # Timestamp
        df['time'] = pd.to_datetime(
            df['YEAR-MM-DD'].astype(str) + ' ' + df['HR:MN:SS.SS'].astype(str),
            errors='coerce', format='%Y-%m-%d %H:%M:%S.%f'
        )

        # Station name
        if 'STN' not in df.columns or df['STN'].isna().all():
            df['STN'] = station_hint if station_hint else pos_path.parent.name

        keep = ['STN','time','lat_deg','lon_deg','HGT(m)']
        return df[keep].dropna(subset=['time','lat_deg','lon_deg'])
    except Exception as e:
        print(f"Error parsing {pos_path}: {e}")
        return pd.DataFrame()

def aggregate_daily_mean(df):
    """Mean daily lat/lon/height; also return count & std for QA."""
    if df.empty:
        return df
    df = df.copy()
    df['date'] = df['time'].dt.date
    g = df.groupby('date').agg({
        'lat_deg': ['mean','std','count'],
        'lon_deg': ['mean','std'],
        'HGT(m)':  ['mean','std']
    })
    g.columns = ['lat_mean','lat_std','count','lon_mean','lon_std','h_mean','h_std']
    return g.reset_index()

def lonlat_to_local_EN(daily_df):
    """
    Approximate local East/North offsets (meters) relative to first day
    using simple equirectangular approximation (good for small displacements).
    """
    if daily_df.empty:
        return daily_df
    lat0 = np.deg2rad(daily_df['lat_mean'].iloc[0])
    lon0 = daily_df['lon_mean'].iloc[0]
    lat = daily_df['lat_mean'].to_numpy()
    lon = daily_df['lon_mean'].to_numpy()
    # meters per degree
    m_per_deg_lat = 110540.0  # ~ meridional
    m_per_deg_lon = 111320.0 * np.cos(lat0)  # ~ zonal at lat0
    dE = (lon - lon0) * m_per_deg_lon
    dN = (lat - np.rad2deg(lat0)) * m_per_deg_lat
    out = daily_df.copy()
    out['dE_m'] = dE
    out['dN_m'] = dN
    return out

def plot_track_lonlat(st, daily_df, outdir):
    """Plot lon-vs-lat track with start/end markers."""
    if daily_df.empty:
        return
    plt.figure(figsize=(6,6))
    plt.plot(daily_df['lon_mean'], daily_df['lat_mean'], marker='o', markersize=2, linewidth=1)
    # mark start/end
    plt.scatter(daily_df['lon_mean'].iloc[0], daily_df['lat_mean'].iloc[0], s=30, label='start')
    plt.scatter(daily_df['lon_mean'].iloc[-1], daily_df['lat_mean'].iloc[-1], s=30, label='end')
    plt.xlabel('Longitude (deg)')
    plt.ylabel('Latitude (deg)')
    plt.title(f'{st}: Daily mean position (track)')
    plt.legend(loc='best')
    plt.gca().set_aspect('equal', adjustable='datalim')
    plt.grid(True, alpha=0.3)
    out_path = outdir / f'{st}_track_lonlat.png'
    plt.tight_layout()
    plt.savefig(out_path, dpi=180)
    plt.close()
    print(f'🖼  saved {out_path}')

def plot_displacement_time(st, daily_df_EN, outdir):
    """Plot East/North displacement vs time (meters) relative to first day."""
    if daily_df_EN.empty:
        return
    plt.figure(figsize=(8,4))
    plt.plot(daily_df_EN['date'], daily_df_EN['dE_m'], label='East (m)', linewidth=1)
    plt.plot(daily_df_EN['date'], daily_df_EN['dN_m'], label='North (m)', linewidth=1)
    plt.xlabel('Date')
    plt.ylabel('Displacement (m) relative to first day')
    plt.title(f'{st}: Daily displacement (E/N)')
    plt.legend(loc='best')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    out_path = outdir / f'{st}_displacement_EN.png'
    plt.savefig(out_path, dpi=180)
    plt.close()
    print(f'🖼  saved {out_path}')

# ──────────────────────────────────────────────────────────────────────────────
# MAIN
# ──────────────────────────────────────────────────────────────────────────────
for st in STATIONS:
    station_rows = []

    for year in range(YEAR_START, YEAR_END + 1):
        station_dir = base_dir / f'{year}' / st
        if not station_dir.exists():
            continue
        pos_files = [p for p in station_dir.iterdir() if p.suffix == '.pos' and not p.name.startswith('._')]
        for pos in pos_files:
            df = read_pos_file(pos, station_hint=st)
            if not df.empty:
                station_rows.append(df)

    if not station_rows:
        print(f'⚠️  No POS data for {st}')
        continue

    station_df = pd.concat(station_rows, ignore_index=True).sort_values('time')

    # daily means
    daily = aggregate_daily_mean(station_df)
    daily.insert(0, 'STN', st)

    # write CSV
    out_csv = OUT_CSV_DIR / f'{st}_daily_mean_{YEAR_START}_{YEAR_END}.csv'
    daily.to_csv(out_csv, index=False, float_format='%.9f')
    print(f'✅ wrote {len(daily):,} rows → {out_csv}')

    # plots
    plot_track_lonlat(st, daily, OUT_FIG_DIR)
    daily_EN = lonlat_to_local_EN(daily)
    plot_displacement_time(st, daily_EN, OUT_FIG_DIR)
